# Identifying the Most Successful Types of Movies for Disney
## Final Project for _Programming in Python For Data Science_
### Data Analyst: Cameron Turner
### Date: December 19, 2022

# Introduction

## Questions of Interest

Disney has made hundreds of movies over the past several decades. Known for their G-rated family films, they have also created numerous other movies in other genres that appeal to a wide range of audiences.  

What I'm interested in determining, and is the focus of this analysis, is to:

* identify which types of movies have been the most successful for Disney; and 
* identify the type of movie on which Disney should focus in the future

Disney's stock is down 38% year-to-date, 2022 (Google Finance).  With the recent return of ex-Disney CEO, Bob Iger, to the top spot, the information in this analysis is important as one of his first priorities is to look at Disney's cost models across its businesses (F Pallotta, CNN).  In addition to Disney+, this would most likely also involve adjusting and streamlining Disney's overall entertainment strategy including its films. 

Given the focus of Disney's theme parks, I would suspect that their family-friendly G-rated movies would do well in this analysis. 

## Dataset Description

The dataset used in this analysis is the *Disney Character Success* dataset compiled by Kelly Garrett.  It can be found here:

https://data.world/kgarrett/disney-character-success-00-16

The dataset contains five tables found in the following files: `disney_movies_total_gross.csv`, `disney_revenue_1991-2016.csv`, `disney-characters.csv`, `disney-director.csv` and `disney-voice-actors.csv`.

For this analysis, the _disney_movies_total_gross.csv_ and _disney_revenue_1991-2016.csv_ datasets will be used.  

* _disney_movies_total_gross.csv_: this dataset contains data on movie titles, release dates, genre, ratings, total gross revenue and total gross revenue adjusted for inflation.  
* _disney_revenue_1991-2016.csv_: this dataset contains data on how much revenue was made each year from 1991 - 2016 in the areas of Studio Entertainment, Consumer Products, Disney Parks and Total Revenue.

# Methods and Results

First things first.  In order to start our analysis, we need to import the required libraries.  Let's import the Pandas library for data wrangling / analysis and the Altair library for data visualization.

In [1]:
import pandas as pd
import altair as alt


In order to analyze the Disney data and answer the question, we'll need to load in the following two CSV files:

* disney_movies_total_gross.csv
* disney_revenue_1991-2016.csv

Let's begin!

In [2]:
all_movies_df  = pd.read_csv("data/disney_movies_total_gross.csv")
all_revenue_df = pd.read_csv("data/disney_revenue_1991-2016.csv")


After loading in any CSV file, we're always excited to see what the data looks like! Let's take a look at some sample data in the CSV files.  Starting with the *disney_movies_total_gross.csv*.

In [3]:
all_movies_df.head()

,movie_title,release_date,genre,MPAA_rating,total_gross,inflation_adjusted_gross
0,Snow White and the Seven Dwarfs,"Dec 21, 1937",Musical,G,"$184,925,485","$5,228,953,251"
1,Pinocchio,"Feb 9, 1940",Adventure,G,"$84,300,000","$2,188,229,052"
2,Fantasia,"Nov 13, 1940",Musical,G,"$83,320,000","$2,187,090,808"
3,Song of the South,"Nov 12, 1946",Adventure,G,"$65,000,000","$1,078,510,579"
4,Cinderella,"Feb 15, 1950",Drama,G,"$85,000,000","$920,608,730"



Hmmm... we'll have to confirm in the dataset description but the *total_gross* and *inflation_adjusted_gross* columns have dollar signs and commas and are most likely stored as *Strings*.  In order for calculations to be performed on these fields, they will need to be converted to *ints*.

Now, let's take a look at the *disney_revenue_1991-2016.csv* dataset.

In [4]:
all_revenue_df.head()

,Year,Studio Entertainment[NI 1],Disney Consumer Products[NI 2],Disney Interactive[NI 3][Rev 1],Walt Disney Parks and Resorts,Disney Media Networks,Total
0,1991,2593.0,724.0,NaN,2794.0,NaN,6111
1,1992,3115.0,1081.0,NaN,3306.0,NaN,7502
2,1993,3673.4,1415.1,NaN,3440.7,NaN,8529
3,1994,4793.0,1798.2,NaN,3463.6,359,10414
4,1995,6001.5,2150.0,NaN,3959.8,414,12525


Interesting, again!  The data types for these columns seem to be *floats* and *ints* and the values look to represent *millions* of dollars.  For example, the Total for 1991 is 6111 million dollars which is: $ $6,111,000,000  or  $6.111 billion dollars.  These values will need to be converted into a format compatible with the *gross* revenue numbers in the *disney_movies_total_gross.csv* dataset.

Now that we have had a preview of the data, let's take a look at the datasets to confirm how they're structured.

In [5]:
all_movies_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 579 entries, 0 to 578
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   movie_title               579 non-null    object
 1   release_date              579 non-null    object
 2   genre                     562 non-null    object
 3   MPAA_rating               523 non-null    object
 4   total_gross               579 non-null    object
 5   inflation_adjusted_gross  579 non-null    object
dtypes: object(6)
memory usage: 27.3+ KB


The **disney_movies_total_gross.csv** dataset has 6 fields containing 579 records.  All of the columns are of an *object* data type which was suspected when the sample data was previewed above.  

Each movie consists of a **movie_title**, a **release_date**, a **genre**, an **MPAA_rating**, the **total_gross** (earnings) and the **inflation_adjusted_gross** (earnings in today's dollars).  The fields that look to be the most relevant to answer our question will be:

* release_date
* genre
* MPAA_rating
* total_gross
* inflation_adjusted_gross

Now, let's take a look at the revenue CSV file.

In [6]:
all_revenue_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 7 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Year                             26 non-null     int64  
 1   Studio Entertainment[NI 1]       25 non-null     float64
 2   Disney Consumer Products[NI 2]   24 non-null     float64
 3   Disney Interactive[NI 3][Rev 1]  12 non-null     float64
 4   Walt Disney Parks and Resorts    26 non-null     float64
 5   Disney Media Networks            23 non-null     object 
 6   Total                            26 non-null     int64  
dtypes: float64(4), int64(2), object(1)
memory usage: 1.5+ KB


The **disney_revenue_1991-2016.csv** dataset has 7 fields that are a combination of *int*, *float* and *object* data types.  There are 26 records from, as is indicated in the dataset title, the years 1991 - 2016.  The fields that look to be the most relevant to answer our questions will be:

* Year
* Studio Entertainment[NI 1]
* Total

**Note:** The *Disney Media Networks* column is an _object_ data type but since it's not going to be used to answer our questions, it will be left alone.

### Data Cleaning and Preparation

Based on reviewing the dataset types and data above, it's time to start cleaning the data and preparing it for analysis.

First of all, convert the *release_date* in the *disney_movies_total_gross.csv* to a _DateTime_ data type.

In [7]:
all_movies_df = all_movies_df.assign(release_date = pd.to_datetime(all_movies_df['release_date']))


Let's make sure it worked!

In [8]:
all_movies_df.dtypes

movie_title                         object
release_date                datetime64[ns]
genre                               object
MPAA_rating                         object
total_gross                         object
inflation_adjusted_gross            object
dtype: object

Good!  Let's now take a look to quickly guage the completeness of the dataset and see what values are null.

In [9]:
all_movies_df.isnull().sum()

movie_title                  0
release_date                 0
genre                       17
MPAA_rating                 56
total_gross                  0
inflation_adjusted_gross     0
dtype: int64

With 579 total records, the data is looking pretty complete.  As we are only going to be interested in the years from 1991 - 2016 to match the other dataset, we'll re-calculate the null values after we have reduced the dataset.

Now, let's convert the __total_gross__ and __inflation_adjusted_gross__ fields to _ints_.  First of all, split the fields out by the $.

In [10]:
tgd_df = all_movies_df['total_gross'].str.split('$', expand=True).rename(columns={1:'total_gross_dollars'})
iag_df = all_movies_df['inflation_adjusted_gross'].str.split('$', expand=True).rename(columns={1:'inflation_adjusted_gross_dollars'})



And now replace the commas with an empty String and change them to the _int_ data type.

In [11]:
# Use chaining to remove the commas and convert them from objects to int in a single line.
#
tgd_df['total_gross_dollars'] = tgd_df['total_gross_dollars'].str.replace(',', '').astype('int')
iag_df['inflation_adjusted_gross_dollars'] = iag_df['inflation_adjusted_gross_dollars'].str.replace(',', '').astype('int')


Let's concatenate these two newly created fields to the dataframe.

In [12]:
# Concat these two fields to the dataframe.
#
all_movies_df = pd.concat([all_movies_df, tgd_df[['total_gross_dollars']], iag_df[['inflation_adjusted_gross_dollars']]], axis=1)

And let's verify that it worked.

In [13]:
all_movies_df.head()

,movie_title,release_date,genre,MPAA_rating,total_gross,inflation_adjusted_gross,total_gross_dollars,inflation_adjusted_gross_dollars
0,Snow White and the Seven Dwarfs,1937-12-21,Musical,G,"$184,925,485","$5,228,953,251",184925485,5228953251
1,Pinocchio,1940-02-09,Adventure,G,"$84,300,000","$2,188,229,052",84300000,2188229052
2,Fantasia,1940-11-13,Musical,G,"$83,320,000","$2,187,090,808",83320000,2187090808
3,Song of the South,1946-11-12,Adventure,G,"$65,000,000","$1,078,510,579",65000000,1078510579
4,Cinderella,1950-02-15,Drama,G,"$85,000,000","$920,608,730",85000000,920608730


Looks good!  To clean our dataframe up, let's drop those original columns as they no longer serve a purpose.

In [14]:

all_movies_df = all_movies_df.drop(columns=['total_gross', 'inflation_adjusted_gross'])

Before we check to ensure those columns are gone, let's create a __year__ column to be the _Year_ value of the __release_date__ in order for it to align with the _Year_ column in the __disney_revenue_1991-2016.csv__ dataset.  This is where it was valuable to have converted the __release_date__ field to a _DateTime_ type.

In [15]:

all_movies_df['Year'] = all_movies_df['release_date'].dt.year


Now that we've dropped two columns and added a new column, __Year__, based on the __release_date__, let's take a look at the dataframe structure.

In [16]:
all_movies_df.dtypes


movie_title                                 object
release_date                        datetime64[ns]
genre                                       object
MPAA_rating                                 object
total_gross_dollars                          int64
inflation_adjusted_gross_dollars             int64
Year                                         int64
dtype: object

Looks good!  The two original __total_gross__ and __inflation_adjusted_gross__ columns are gone, the __release_date__ column is still a _DateTime_ and now a new column, __Year__, is an _int_ as it is just the Year.  Let's take a peek at the data.

In [17]:
all_movies_df.head()

,movie_title,release_date,genre,MPAA_rating,total_gross_dollars,inflation_adjusted_gross_dollars,Year
0,Snow White and the Seven Dwarfs,1937-12-21,Musical,G,184925485,5228953251,1937
1,Pinocchio,1940-02-09,Adventure,G,84300000,2188229052,1940
2,Fantasia,1940-11-13,Musical,G,83320000,2187090808,1940
3,Song of the South,1946-11-12,Adventure,G,65000000,1078510579,1946
4,Cinderella,1950-02-15,Drama,G,85000000,920608730,1950


Perfect!  This is what we're looking for.  There is just one last thing to do to prepare this dataset for analysis and that is to remove all records that are earlier than 1991.  This is because our other dataset only contains data from 1991 onwards so that is the subset of data we're going to use.  Let's create a new dataframe with just this data.

In [18]:

movies_df = all_movies_df[all_movies_df['Year'] > 1990]

Let's do a quick check to see what the minimum year is in this new dataframe.

In [19]:
movies_df['Year'].min()

1991

Excellent!  To finish, let's take a look at how many records are left and how many are null.

In [20]:
movies_df.shape[0]

479

In [21]:
movies_df.isnull().sum()


movie_title                          0
release_date                         0
genre                               11
MPAA_rating                          5
total_gross_dollars                  0
inflation_adjusted_gross_dollars     0
Year                                 0
dtype: int64

As there are 11 movies missing a __genre__ and 5 __MPAA_rating__ out of 479 (3.3% if combined) and it would be very difficult to impute values for these based on the movie's title, let's just drop them from the analysis.

In [22]:
movies_df = movies_df.dropna()

Let's see how many movies were dropped.

In [23]:
movies_df.shape[0]

464

Therefore, 15 movies or (3.1%) of the dataset were dropped.  That sounds good to me!

Now let's work on the __disney_revenue_1991-2016.csv__ dataset which is in the __all_revenue_df__ dataframe.

From earlier, the __Studio Entertainment[NI 1]__ column was one of interest.  However, let's rename that field go get rid of the _[NI 1]_. 

In [24]:
all_revenue_df = all_revenue_df.rename(columns={'Studio Entertainment[NI 1]' : 'Studio Entertainment'})


Let's check to make sure that worked but let's also quickly see what's the data completeness by checking to see what data is null at the same time.

In [25]:
all_revenue_df.isnull().sum().sort_values(ascending=False)

Disney Interactive[NI 3][Rev 1]    14
Disney Media Networks               3
Disney Consumer Products[NI 2]      2
Studio Entertainment                1
Year                                0
Walt Disney Parks and Resorts       0
Total                               0
dtype: int64

As we're only interested in the columns, __Studio Entertainment__, __Year__ and __Total__, there is only one value that's missing in __Studio Entertainment__.  Let's just fill this empty value with the value of its predecessor.  This will, in fact, give it a conservative ROI for that year of 0%.

In [26]:
all_revenue_df = all_revenue_df.fillna(method='ffill')


In [27]:
all_revenue_df

,Year,Studio Entertainment,Disney Consumer Products[NI 2],Disney Interactive[NI 3][Rev 1],Walt Disney Parks and Resorts,Disney Media Networks,Total
0,1991,2593.0,724.0,NaN,2794.0,NaN,6111
1,1992,3115.0,1081.0,NaN,3306.0,NaN,7502
2,1993,3673.4,1415.1,NaN,3440.7,NaN,8529
3,1994,4793.0,1798.2,NaN,3463.6,359,10414
4,1995,6001.5,2150.0,NaN,3959.8,414,12525
5,1996,6001.5,2150.0,NaN,4502.0,"4,142",18739
6,1997,6981.0,3782.0,174.0,5014.0,6522,22473
7,1998,6849.0,3193.0,260.0,5532.0,7142,22976
8,1999,6548.0,3030.0,206.0,6106.0,7512,23402
9,2000,5994.0,2602.0,368.0,6803.0,9615,25402


That looks better!  The __Studio Entertainment__ values for 1995 and 1996 are the same so it must have been 1996 that was missing the data.

Now, let's cleanup the __revenue_df__ dataframe by only keeping the columns in which we're interested:

* Year
* Studio Entertainment
* Total

In [28]:
revenue_df = all_revenue_df.loc[:, ['Year', 'Studio Entertainment', 'Total']]
revenue_df.head()

,Year,Studio Entertainment,Total
0,1991,2593.0,6111
1,1992,3115.0,7502
2,1993,3673.4,8529
3,1994,4793.0,10414
4,1995,6001.5,12525



There is one last thing to do for this dataset.  We need to convert the __Studio Entertainment__ and __Total__ columns to millions to align with the __total_gross_dollars__ in the other dataset.  First of all, let's convert the __Studio Entertainment__ data type from a _float_ to an _int_ to be consistent with our other numbers.

In [29]:
revenue_df['Studio Entertainment'] = revenue_df['Studio Entertainment'].astype('int')


In [30]:
# Multiply the Studio Entertainment and Total columns by a million dollars.
#
million_dollars = 1000000

revenue_df['Studio Entertainment'] = revenue_df['Studio Entertainment'] * million_dollars
revenue_df['Total'] = revenue_df['Total'] * million_dollars

In [31]:
revenue_df.head()

,Year,Studio Entertainment,Total
0,1991,2593000000,6111000000
1,1992,3115000000,7502000000
2,1993,3673000000,8529000000
3,1994,4793000000,10414000000
4,1995,6001000000,12525000000


Looks great!  It is now all ready to be merged in with the other data set.  The type of merge that will be done is an _inner_ merge on the __Year__ column.  Let's first count the number of rows in the base dataframe (_left_).

In [32]:
movies_df.shape[0]

464

There are 464 records.  After the merge, given all the data cleaning that was done previously and that the merge will be an inner merge done on __Year__, there should be approximately 464 records in an inner merge unless of course there are missing years from the range 1991-2016.

In [33]:
movie_merge_df = movies_df.merge(revenue_df, left_on='Year', right_on='Year', how='inner')
movie_merge_df

,movie_title,release_date,genre,MPAA_rating,total_gross_dollars,inflation_adjusted_gross_dollars,Year,Studio Entertainment,Total
0,White Fang,1991-01-18,Adventure,PG,34729091,69540672,1991,2593000000,6111000000
1,Haakon Haakonsen,1991-03-01,Adventure,PG,15024232,30084149,1991,2593000000,6111000000
2,The Marrying Man,1991-04-05,Romantic Comedy,R,12454768,24939118,1991,2593000000,6111000000
3,Oscar,1991-04-26,Comedy,PG,23562716,47181395,1991,2593000000,6111000000
4,One Good Cop,1991-05-03,Action,R,11276846,22580472,1991,2593000000,6111000000
...,...,...,...,...,...,...,...,...,...
459,The Light Between Oceans,2016-09-02,Drama,PG-13,12545979,12545979,2016,9441000000,55632000000
460,Queen of Katwe,2016-09-23,Drama,PG,8874389,8874389,2016,9441000000,55632000000
461,Doctor Strange,2016-11-04,Adventure,PG-13,232532923,232532923,2016,9441000000,55632000000
462,Moana,2016-11-23,Adventure,PG,246082029,246082029,2016,9441000000,55632000000


The merge was a success!  There are still 464 records so there must have been records in both datasets that matched on __Year__.

To complete the data preparation, let's add two more fields:

* studio_contribution_percentage
* total_contribution_percentage

These fields will calculate what percentage of Disney's totals (__Studio Entertainment__ and __Total__) for the release years to which the movie's __total_gross__ contributed.  As these are % of snapshots in time, use the __total_gross_dollars__ field and not the __inflation_adjusted_gross_dollars__ field.

In [34]:
movie_merge_df = movie_merge_df.assign(studio_contribution_percentage = (movie_merge_df['total_gross_dollars'] / movie_merge_df['Studio Entertainment'])*100)
movie_merge_df = movie_merge_df.assign(total_contribution_percentage = (movie_merge_df['total_gross_dollars'] / movie_merge_df['Total'])*100)

movie_merge_df.head()

,movie_title,release_date,genre,MPAA_rating,total_gross_dollars,inflation_adjusted_gross_dollars,Year,Studio Entertainment,Total,studio_contribution_percentage,total_contribution_percentage
0,White Fang,1991-01-18,Adventure,PG,34729091,69540672,1991,2593000000,6111000000,1.339340,0.568305
1,Haakon Haakonsen,1991-03-01,Adventure,PG,15024232,30084149,1991,2593000000,6111000000,0.579415,0.245856
2,The Marrying Man,1991-04-05,Romantic Comedy,R,12454768,24939118,1991,2593000000,6111000000,0.480323,0.203809
3,Oscar,1991-04-26,Comedy,PG,23562716,47181395,1991,2593000000,6111000000,0.908705,0.385579
4,One Good Cop,1991-05-03,Action,R,11276846,22580472,1991,2593000000,6111000000,0.434896,0.184534


There!  I think we're all done with the data preparation.  To recap, the following data preparation techniques were used:

* column data type conversions
* column heading replacement
* column removals
* column splitting and concatenation
* drop null values
* impute null values (ffill)
* dataframe merging
* column creation with data calculations



### Data Analysis

Once the data has been wrangled and cleaned, let's begin the data analysis.

To start, a helper Python script was created called __chart_maker.py__  It contains the following three functions that will help us in our analysis:

* create_dataframe_group_by_columns_and_agg
* plot_barchart
* plot_scatter_chart

These functions will help group and visualize our data.  

Let's first run our test script to make sure our primary function, _create_dataframe_group_by_columns_and_agg_ is working as expected.

In [35]:
from test_chart_maker import test_create_dataframe_group_by_columns_and_agg

test_create_dataframe_group_by_columns_and_agg()

Table 1 : count on movie_title for the group ['genre']
Table 2 : min on year for the group ['genre']
Table 3 : max on year for the group ['genre']
Table 4 : sum on year for the group ['genre']


If this point has been reached and there is no error, then everything is good! All assertions passed.

Let's now move on to importing the _create_dataframe_group_by_columns_and_agg_ function and doing a general count of movies based on __genre__.  

In [36]:
from chart_maker import create_dataframe_group_by_columns_and_agg

create_dataframe_group_by_columns_and_agg(movie_merge_df, ['genre'], 'movie_title', 'count')

Table 5 : count on movie_title for the group ['genre']


,movie_title
genre,
Comedy,142
Adventure,111
Drama,92
Action,34
Thriller/Suspense,22
Romantic Comedy,20
Documentary,16
Musical,10
Western,7


The genres, Comedy, Adventure and Drama seem to be the most popular types of movies made by Disney with most movies being made from 1991-2016 being in the _Comedy_ category.  Let's visualize this in a graph using another one of our functions.

Let's import the _plot_barchart_ function to visualize the data in a bar chart.

In [37]:
from chart_maker import plot_barchart

my_graph = plot_barchart(movie_merge_df, 'Number of Movies by Genre', 'genre', 'Genre', 'movie_title', 'Movies', 'count')
my_graph


alt.Chart(...)

Of course, no surprise based on the previous information, but a visualization gives a better sense of scale.  Drama, Adventure and Comedy are by far the most made movies by Disney, far more than the remaining genres combined.  Therefore, let's check to see if more movies translate into more film revenue.




In [38]:
create_dataframe_group_by_columns_and_agg(movie_merge_df, ['genre'], 'total_gross_dollars', 'sum')

Table 6 : sum on total_gross_dollars for the group ['genre']


,total_gross_dollars
genre,
Adventure,15677624824
Comedy,6520603755
Action,4003990137
Drama,3471135652
Thriller/Suspense,1340891861
Romantic Comedy,912372585
Musical,663430923
Western,359011459
Documentary,180685619


Now that's interesting.  There were less than half the Action films created than Drama films yet Action films earned more.  Thriller movies also do well but they are far behind the fourth place Drama genre.

**Data Analysis Decision:** focus on Adventure, Comedy, Action and Drama movies and discard all of the other genres as they don't generate the revenue as these other four.


In [39]:
movie_merge_df = movie_merge_df[(movie_merge_df['genre'] == 'Comedy') | (movie_merge_df['genre'] == 'Adventure') | (movie_merge_df['genre'] == 'Drama') | (movie_merge_df['genre'] == 'Action')]

Now that we're just dealing with the big four genres, let's see how much these movies grossed over the years.  Let's import the _plot_scatter_chart_ function from the _chart_maker_ script.

In [40]:
from chart_maker import plot_scatter_chart

my_graph = plot_scatter_chart(movie_merge_df, 'Year', 'total_gross_dollars', 'Total Gross Revenue by Year')
my_graph


alt.Chart(...)

Most movies over the years earn in the 100M range but as the years progress more and more of them start earning larger sums.

Let's see how much they grossed by genre over the years.

In [41]:
create_dataframe_group_by_columns_and_agg(movie_merge_df, ['genre'], 'total_gross_dollars', 'sum')

Table 7 : sum on total_gross_dollars for the group ['genre']


,total_gross_dollars
genre,
Adventure,15677624824
Comedy,6520603755
Action,4003990137
Drama,3471135652


And as a graph...

In [42]:
my_graph = plot_barchart(movie_merge_df, 'Total Movie Gross Revenue by Genre', 'genre', 'Genre', 'total_gross_dollars', 'Gross Revenue', 'sum')
my_graph


alt.Chart(...)

Interesting! This is a good insight - Disney made more Comedy movies between 1991-2016 but it was their Adventure movies (which was 2nd in overall number of movies made) that grossed the most revenue.  Drama, while significant, is a smaller amount.  

Let's see what happens when we use the __inflation_adjusted_gross_dollars__.

In [43]:
create_dataframe_group_by_columns_and_agg(movie_merge_df, ['genre'], 'inflation_adjusted_gross_dollars', 'sum')

Table 8 : sum on inflation_adjusted_gross_dollars for the group ['genre']


,inflation_adjusted_gross_dollars
genre,
Adventure,19475539485
Comedy,10321988303
Action,5112927313
Drama,5042060133


And as a graph...

In [44]:
my_graph = plot_barchart(movie_merge_df, 'Inflation Adjusted Movie Gross Revenue by Genre', 'genre', 'Genre', 'inflation_adjusted_gross_dollars', 'Gross Revenue', 'sum')
my_graph


alt.Chart(...)

This graph looks similar to the last one with similar relative values.

Let's plot a similar graph to see which genre had, on average, the best showing at the box-office.


In [45]:
my_graph = plot_barchart(movie_merge_df, 'Average Movie Gross Revenue by Genre', 'genre', 'Genre', 'total_gross_dollars', 'Gross Revenue', 'mean')
my_graph


alt.Chart(...)

This is even more interesting.  On average, the Adventure movies made just over 140M per film compared to Drama films which made just under 40M per film.  Comedy films earn just over 40M per film.  Even though Disney made more Comedy movies, they earn more per film on their Adventure movies.  

The sleeper hit is that even though there were only 34 Action films made, they earned on average just under 120M per film.  Let's take a look at the maximum.


In [46]:
my_graph = plot_barchart(movie_merge_df, 'Maximum Movie Gross Revenue by Genre', 'genre', 'Genre', 'total_gross_dollars', 'Gross Revenue', 'max')
my_graph


alt.Chart(...)

When the maximum movie gross revenue is taken into consideration, Action movies have great blockbuster potential like Adventure movies.  Far better than their Comedy and Drama movies.

Finally, let's plot a scatter plot of __total_gross_dollars__ and __inflation_adjusted_gross_dollars__ to determine if there is any correlation.

In [47]:

my_graph = plot_scatter_chart(movie_merge_df, 'total_gross_dollars', 'inflation_adjusted_gross_dollars', 'Adjusted Gross Revenue vs Unadjusted Total Gross Revenue')
my_graph

alt.Chart(...)

When plotting __total_gross_dollars__ vs __inflation_adjusted_gross_dollars__, it appears to have a positively correlated linear relationshiop.

**Data Analysis Decision:** for this analysis, the __total_gross_dollars__ value will most likely give us similar information as __inflation_adjusted_gross_dollars__ when looking to answer our questions.  Let's concentrate on __total_gross_dollars__ from here on out.

Let's drill down a bit more to see the total amounts per __genre__ and __MPAA_rating__ to see if a movie's audience rating has any significance.


In [48]:
create_dataframe_group_by_columns_and_agg(movie_merge_df, ['genre', 'MPAA_rating'], 'movie_title', 'count')

Table 9 : count on movie_title for the group ['genre', 'MPAA_rating']


movie_title
genre     MPAA_rating             
Comedy    PG                    68
Adventure PG                    55
          G                     36
Comedy    PG-13                 35
Drama     PG-13                 34
          R                     29
          PG                    26
Comedy    R                     23
Action    PG-13                 18
Adventure PG-13                 17
Comedy    G                     16
Action    R                     12
          PG                     4
Adventure R                      3
Drama     G                      3

In the land of Disney, PG-rated movies reign supreme followed by PG-13.  Interestingly enough, G-rated movies, even though it has a top three showing, it didn't have quite as strong a showing as one may have thought.

Let's see how these movies fared when looking at total gross revenue per genre and MPAA rating.

In [49]:
create_dataframe_group_by_columns_and_agg(movie_merge_df, ['genre', 'MPAA_rating'], 'total_gross_dollars', 'sum')

Table 10 : sum on total_gross_dollars for the group ['genre', 'MPAA_rating']


total_gross_dollars
genre     MPAA_rating                     
Adventure PG                    7205710759
          G                     4397262384
          PG-13                 3976887642
Comedy    PG                    3391817381
Action    PG-13                 3146937783
Comedy    G                     1561368699
Drama     PG                    1417638068
          PG-13                 1343360585
Comedy    PG-13                 1197235101
Action    R                      748861460
Drama     R                      548762872
Comedy    R                      370182574
Drama     G                      161374127
Action    PG                     108190894
Adventure R                       97764039

Apart from R-rated Adventure movies, the Adventure movies rated as G, PG and PG-13 are the top money makers for Disney from 1991 - 2016.  Comedy and Action movies do pretty well for PG and PG-13.  Also interesting is that the R-rated movies tend to gross the least amount of revenue relative to the other movies.  This could be due to the lack of number of movies made in this category (52 / 484 = 10.7%) but most likely it's because they don't have the widest available audience like the other rated movies due to age restrictions.

Let's see what the data looks like when we account for inflation which can also verify our previous decision to discard it.

In [50]:
create_dataframe_group_by_columns_and_agg(movie_merge_df, ['genre', 'MPAA_rating'], 'inflation_adjusted_gross_dollars', 'sum')

Table 11 : sum on inflation_adjusted_gross_dollars for the group ['genre', 'MPAA_rating']


inflation_adjusted_gross_dollars
genre     MPAA_rating                                  
Adventure PG                                 8321492545
          G                                  6451379879
Comedy    PG                                 5517366281
Adventure PG-13                              4550807668
Action    PG-13                              3486948398
Comedy    G                                  2333273694
Drama     PG                                 1922733926
          PG-13                              1907521153
Comedy    PG-13                              1831439437
Action    R                                  1407044726
Drama     R                                  1002558604
Comedy    R                                   639908891
Action    PG                                  218934189
Drama     G                                   209246450
Adventure R                                   151859393

When we  we account for inflation, there is a slight change than when using __total_gross__ but otherwise, the top five look the same.  The decision to discard __inflation_adjusted_gross_dollars__ seems to be sound.

**Data Analysis Decision:** focus on PG, PG-13 and G rated movies.

Let's now do some analyses by __Year__ starting with counting how many films of each genre were released each year.

In [51]:
create_dataframe_group_by_columns_and_agg(movie_merge_df, ['Year', 'genre'], 'movie_title', 'count')

Table 12 : count on movie_title for the group ['Year', 'genre']


,,movie_title
Year,genre,
1994,Comedy,13
1992,Comedy,11
1997,Comedy,11
1995,Drama,10
2016,Adventure,9
...,...,...
2000,Action,1
2006,Action,1
2010,Comedy,1


Looking at individual years, Disney released a lot of Comedy films in the 1990's.

In [52]:
create_dataframe_group_by_columns_and_agg(movie_merge_df, ['Year', 'genre'], 'total_gross_dollars', 'sum')

Table 13 : sum on total_gross_dollars for the group ['Year', 'genre']


,,total_gross_dollars
Year,genre,
2016,Adventure,2408423122
2015,Adventure,1522076961
2010,Adventure,1185280338
2013,Adventure,1110359474
2003,Adventure,886400062
...,...,...
1992,Action,29028000
2010,Comedy,25702053
2016,Drama,21420368


When looking at total gross dollars by year, Adventure movies account for the top five.  Action or Comedy films don't make the top five.  

Continuing with an analysis including __Year__, let's find out what type of movie and rating had the highest revenue per year.

In [53]:
create_dataframe_group_by_columns_and_agg(movie_merge_df, ['Year', 'genre', 'MPAA_rating'], 'total_gross_dollars', 'max')


Table 14 : max on total_gross_dollars for the group ['Year', 'genre', 'MPAA_rating']


total_gross_dollars
Year genre     MPAA_rating                     
2015 Adventure PG-13                  936662225
2012 Action    PG-13                  623279547
2016 Adventure PG-13                  529483936
               PG                     486295561
2015 Action    PG-13                  459005868
...                                         ...
2013 Drama     R                        3254172
1997 Drama     PG                       1775644
2003 Drama     R                        1569918
2010 Adventure PG-13                      48658
1998 Comedy    R                          45779

[202 rows x 1 columns]

Adventure and Action PG-13 and PG movies had the highest grossing films in different years.  Drama, on the other hand, had some of the lowest.

Let's now move to start looking at each genre's average contribution value to the Studio revenue and the total Disney revenue by year.

In [54]:
create_dataframe_group_by_columns_and_agg(movie_merge_df, ['Year', 'genre'], 'studio_contribution_percentage', 'mean')


Table 15 : mean on studio_contribution_percentage for the group ['Year', 'genre']


studio_contribution_percentage
Year genre                                    
2012 Action                          10.700078
2013 Action                           6.840480
     Adventure                        4.642747
2015 Action                           4.338909
2016 Action                           4.322470
...                                        ...
2001 Drama                            0.241706
1992 Adventure                        0.227914
1997 Drama                            0.183787
2005 Drama                            0.162208
2016 Drama                            0.113443

[98 rows x 1 columns]

When calculating the average contribution to that year's Studio revenue, Action movies now come out on top.  Let's see how they fare when looking at the total Disney revenue for that year.

In [55]:
create_dataframe_group_by_columns_and_agg(movie_merge_df, ['Year', 'genre'], 'total_contribution_percentage', 'mean')


Table 16 : mean on total_contribution_percentage for the group ['Year', 'genre']


,,total_contribution_percentage
Year,genre,
1994,Adventure,1.523192
2012,Action,1.474241
2013,Action,0.908044
1991,Comedy,0.780321
1992,Comedy,0.762263
...,...,...
2001,Drama,0.065642
2008,Drama,0.058276
1997,Drama,0.057091


This is interesting.  When we look at average contribution value based on the __total_contribution_percentage__, there is a different top 5 with Comedy films actually faring pretty well.  Drama on the other hand, not so much!

Let's just quickly see if there is a positive correlation between the studio and total contributions.  This will help to validate that we would get similar relative numbers regardless of which contribution percentage we used.

In [56]:
my_graph = plot_scatter_chart(movie_merge_df, 'total_contribution_percentage', 'studio_contribution_percentage', 'Studio Contribution % vs Total Contribution %')
my_graph

alt.Chart(...)

When plotting __studio_contribution_percentage__ vs __total_contribution_percentage__, it appears that there is a positively correlated linear relationship.

Let's see if there is any difference to the contributing genres of movies when we include __MPAA_rating__.

In [57]:
create_dataframe_group_by_columns_and_agg(movie_merge_df, ['Year', 'genre', 'MPAA_rating'], 'total_contribution_percentage', 'mean')



Table 17 : mean on total_contribution_percentage for the group ['Year', 'genre', 'MPAA_rating']


,,,total_contribution_percentage
Year,genre,MPAA_rating,
1994,Adventure,G,4.059729
2015,Adventure,PG-13,1.785309
1992,Comedy,G,1.630443
2012,Action,PG-13,1.474241
1995,Adventure,G,1.330842
...,...,...,...
1997,Drama,PG,0.007901
2013,Drama,R,0.007225
2003,Drama,R,0.005801


When we analyze to include __MPAA_rating__, there is a slight difference in the top five but still, Adventure G and PG-13 movies reign.

**Data Analysis Decision:** movies rated G, PG or PG-13 fare the best.  Therefore, R-rated movies should be minimized as they don't provide the same value as the others.  Also, when comparing contributions, it doesn't seem to matter if the Studio or Total values are used as the relative comparisons would be similar.



Lastly, let's take a look at recent movies (i.e., 2012 - 2016) to see what types of movies were contributing the most to the __studio_contribution_percentage__ and are the movies that have been the most successful for Disney recently.

In [58]:
recent_movies_df = movie_merge_df[movie_merge_df['Year'] > 2011]

In [59]:
create_dataframe_group_by_columns_and_agg(recent_movies_df, ['Year', 'genre'], 'studio_contribution_percentage', 'sum')


Table 18 : sum on studio_contribution_percentage for the group ['Year', 'genre']


studio_contribution_percentage
Year genre                                    
2016 Adventure                       25.510254
2015 Adventure                       20.663548
2013 Adventure                       18.570990
2014 Adventure                       11.765114
2012 Action                          10.700078
     Adventure                        8.908945
2015 Action                           8.677817
2013 Action                           6.840480
2016 Action                           4.322470
2012 Drama                            4.231643
2014 Action                           4.167566
2015 Drama                            3.334668
2013 Comedy                           2.022786
2014 Comedy                           1.623153
2013 Drama                            1.534635
2012 Comedy                           0.605799
2014 Drama                            0.500796
2016 Drama                            0.226887

As a sum of contribution for each year, the biggest contributors to the studio revenue were from Adventure movies.

Let's take a look at the biggest contributors on average.

In [60]:
create_dataframe_group_by_columns_and_agg(recent_movies_df, ['Year', 'genre'], 'studio_contribution_percentage', 'mean')


Table 19 : mean on studio_contribution_percentage for the group ['Year', 'genre']


studio_contribution_percentage
Year genre                                    
2012 Action                          10.700078
2013 Action                           6.840480
     Adventure                        4.642747
2015 Action                           4.338909
2016 Action                           4.322470
2015 Adventure                        4.132710
2014 Adventure                        2.941279
2016 Adventure                        2.834473
2012 Adventure                        2.227236
2014 Action                           2.083783
2015 Drama                            1.667334
2012 Drama                            1.410548
2013 Comedy                           1.011393
2014 Comedy                           0.811576
2012 Comedy                           0.605799
2013 Drama                            0.511545
2014 Drama                            0.500796
2016 Drama                            0.113443

Interesting, when we look at which genres are the recent biggest contributors to Studio revenue on average, it is the Action films.

And lastly, let's take a look at the biggest contributors on average by __MPAA_rating__.

In [61]:
create_dataframe_group_by_columns_and_agg(recent_movies_df, ['Year', 'genre', 'MPAA_rating'], 'studio_contribution_percentage', 'mean')


Table 20 : mean on studio_contribution_percentage for the group ['Year', 'genre', 'MPAA_rating']


studio_contribution_percentage
Year genre     MPAA_rating                                
2015 Adventure PG-13                             12.716023
2012 Action    PG-13                             10.700078
2013 Action    PG-13                              6.840480
     Adventure PG                                 5.314509
2014 Adventure PG-13                              4.577798
2013 Adventure G                                  4.490522
2015 Action    PG-13                              4.338909
2016 Action    PG-13                              4.322470
     Adventure PG-13                              4.035679
2012 Adventure PG                                 3.662617
2013 Adventure PG-13                              3.451449
2016 Adventure PG                                 2.491271
2014 Adventure PG                                 2.395772
     Action    PG-13                              2.083783
2015 Adventure PG                                 1.986881
2012 Drama     PG-13                              1.670728
2015 Drama     PG                                 1.667334
2013 Comedy    PG                                 1.509995
2012 Adventure PG-13                              1.254226
     Drama     PG                                 0.890188
2014 Comedy    PG                                 0.811576
2013 Drama     PG-13                              0.740104
2012 Comedy    PG                                 0.605799
2013 Comedy    PG-13                              0.512792
2014 Drama     PG                                 0.500796
2012 Adventure G                                  0.329485
2016 Drama     PG-13                              0.132888
               PG                                 0.093998
2013 Drama     R                                  0.054427

In the past five years, the films that contributed the most to the studio revenue were Adventure and Action PG-13 movies.

I think we now have all the information we need!

## Conclusion

The purpose of this analysis was to answer the following two questions:

* identify which types of movies have been the most successful for Disney; and 
* identify which type of movie on which Disney should focus in the future

To answer these questions, exploratory data analysis was performed that reviewed and summarized the data, cleaned it and prepared it for analysis.  Analysis was then conducted on the clean, prepared dataset using groupings, aggregate functions (e.g., count, sum, max etc.), data wrangling and data visualization.

In the end, the types of movies that have been the most successful for Disney were determined to have been Adventure and Action movies geared towards PG and PG-13 audiences.  These films generate the most revenue and contribute the most to Disney's bottom line each year.  Drama and Comedy movies were also significant but did not provide the same impact as Adventure and Action movies.  R-rated movies were also determined not to have a significant impact on the results.  Originally, I thought that Disney's family-friendly G-rated movies would be more successful but the analysis shows they are not as successful as the PG and PG-13 movies.

As for which types of movies Disney should focus on in the future, the Action / Adventure types of films should definitely be continued to be made as they have been found to be the top grossing money makers in recent years.  Also, there were only 34 Action films made compared to 111 Adventure films but they generate a large amount of revenue on average second only to Adventure films.  Therefore, there looks like there could be an opportunity to increase production of these films and leverage their high average box office receipts.

The impact that these findings could have is to help streamline and focus the types of films that Disney creates and funds in the future.  Disney creates a wide variety of different genres of films, geared towards different ranges of audiences.  While different genres may be moderately successful, Disney has specialized in creating highly successful Adventure and Action films that appeal to a good cross-section of the population (i.e., PG and PG-13), especially to those teenagers / young adults with money to spend!

## References

* Disney Character Success dataset compiled by Kelly Garrett.

    https://data.world/kgarrett/disney-character-success-00-16
    
* Frank Pallotta.  "Bob Iger lays out his priorities for Disney as he returns as CEO", CNN, November 28, 2022

    https://www.cnn.com/2022/11/28/media/bob-iger-disney-town-hall/index.html
    
* Google Finance (Disney Stock)

    https://www.google.com/finance/quote/DIS:NYSE?sa=X&ved=2ahUKEwimqaLM7IT8AhXMxikDHaghBKsQ_AUoAXoECAEQAw&window=1Y
    
